In [1]:
import warnings

# Mute all warnings globally
warnings.filterwarnings("ignore")

# Dynamic Standard Pipeline (user CSVs + NAI) — `DynamicStandardSimGenerator`

Your own inventory CSVs with **NAI-based dynamic harvest**. Mirrors `tests/examples/dynamic_standard_sim_example.py`.

Two modes:

- **NAI from day 1** — omit `end_year` (`years=0`): only spinup runs, then the
  dynamic NAI harvest continues for `dynamic_years`.
- **Static warm-up then NAI** — set `end_year`: a static disturbance schedule
  runs first, then the dynamic phase continues.

## Run in pure-NAI mode

`spinup_warmup_steps` (default 3) settles the AF backward-decomposition DOM pools so the flux series does not start with a spurious negative ramp. Override it via `dynamic_config` if needed.

In [2]:
from goblin_cbm_runner.dynamic_standard_sim_generator import DynamicStandardSimGenerator

gen = DynamicStandardSimGenerator(
    csv_directory='./my_forest/',
    config={'baseline_year': 2020},                              # omit end_year -> years=0
    dynamic_config={'harvest_ratio': 0.75, 'dynamic_years': 30},
)

results = gen.run_flux_simulation()
results.head()

INFO:goblin_cbm_runner.runners.standard_runner:All required files found in: /home/colm/Dropbox/projects/FORESIGHT/packages/cbm_runner/docs/examples/my_forest
INFO:goblin_cbm_runner.runners.standard_runner:Starting DynamicStandard simulation (scenario 0, 2020–2020, years=0)...


Starting AF Simulation...
AF Simulation Complete.
Starting dynamic simulation: 2020 to 2050
  Harvest ratio: 0.75
  Clearfell/thinning split: {'DISTID1': 0.8, 'DISTID2': 0.2}
  Year 2030 complete
  Year 2040 complete
  Year 2050 complete
Dynamic simulation complete: 30 years


,Year,Scenario,AGB,BGB,Deadwood,Litter,Soil,Harvest,Total Ecosystem
0,2021,0,347.031117,77.040908,12.288369,30.066559,3.257072,0.000000,469.684026
1,2022,0,-860.889287,-191.117422,309.033130,428.626882,13.489714,575.889811,-300.856983
2,2023,0,-583.724123,-129.586755,227.693014,259.841134,10.969808,511.243818,-214.806923
3,2024,0,-483.864002,-107.417808,169.517961,147.780400,9.630789,475.103531,-264.352660
4,2025,0,-401.458823,-89.123859,123.644522,70.525604,8.790830,443.739156,-287.621727


In [3]:
gen.export_archive('./dynamic_standard_archive.db')

## Inspect the NAI diagnostics

In [4]:
dr = gen.get_dynamic_result()
print(dr.nai_history.head())
print(dr.sustainability_metrics)

   species   merch_t0   merch_t1  stock_change   harvest       nai  \
0        1  28.411674  31.270243      2.858570  0.000000  2.858570   
1        1  31.270243  27.772091     -3.498152  5.758898  2.260746   
2        1  27.772091  25.841076     -1.931016  5.112438  3.181423   
3        1  25.841076  24.160357     -1.680719  4.751035  3.070316   
4        1  24.160357  22.700482     -1.459875  4.437392  2.977516   

   stand_count  year  volume_targeted  clearfell_stands  thinning_stands  \
0            1  2021         0.428785                 0                1   
1            1  2022         0.339112                 0                1   
2            1  2023         0.477213                 0                1   
3            1  2024         0.460547                 0                1   
4            1  2025         0.446627                 0                1   

   harvest_proportion species_name measurement_type  
0                0.15  Spruce13-16                M  
1             